In [1]:
import sys
import numpy as np
import pandas as pd

from typing import Literal
from collections import Counter
from ase import Atoms
from ase.neighborlist import natural_cutoffs
from ase.visualize import view
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.core.surface import Slab, Structure
from pymatgen.core import Composition
from pymatgen.core.surface import (
    Slab, get_symmetrically_distinct_miller_indices, SlabGenerator)
from ase.neighborlist import NeighborList, natural_cutoffs
from ase.io import read, Trajectory

from gnome_dreams_oer_screening.oer.oer_study_utils import (
    eval_stoichiometry_diff, get_stoichiometry_diff,
    get_stoichiometry_dict)
from gnome_dreams_oer_screening import OER_catalyst_study

ZrO2_m = read('ZrO2_m.cif')
TiO2_r = read('TiO2_rutile.cif')
test_cands = Trajectory('test_cands.traj')

atoms = test_cands[18]
S = AseAtomsAdaptor.get_structure(atoms)

In [2]:
view(ZrO2_m)

<Popen: returncode: None args: ['c:\\Users\\matnis\\OneDrive - Danmarks Tekn...>

In [3]:
cand =  test_cands[18] # TiO2_r
cand

Atoms(symbols='ZnBi3IrRh3O14', pbc=True, cell=[[6.359742925812065, 0.0, 3.677567832273403], [2.11886857481291, 5.771485047280466, 3.6728729851736506], [0.0, 0.0, 7.34648452]])

In [4]:
study = OER_catalyst_study(cand,     
    H2O_gas_free_energy = -14.217,
    H2_gas_free_energy = -6.77,
    use_init_as_relaxed_bulk=True
)
study.bulk_atoms

Atoms(symbols='ZnBi3IrRh3O14', pbc=True, cell=[[6.359742925812065, 0.0, 3.677567832273403], [2.11886857481291, 5.771485047280466, 3.6728729851736506], [0.0, 0.0, 7.34648452]])

In [5]:
study.identify_distinct_surfaces(max_miller=1)
study.get_distinct_millers()

[(1, 1, 1),
 (1, 1, 0),
 (1, 1, -1),
 (1, 0, -1),
 (1, -1, 1),
 (1, -1, 0),
 (0, 1, 0)]

In [6]:
study.predict_most_likely_surfaces()
study.termination_ranking_df.sort_values('Score')
terminations = study.terminations

In [7]:
df = study.get_termination_rankings().sort_values('Score', ascending=False)
df

,Miller indices,Termination index,Score,Normalized score
0,"(1, 1, 1)",0,-0.132885,1.000000
1,"(1, 1, 1)",1,-0.132885,1.000000
13,"(0, 1, 0)",13,-0.149823,0.801596
12,"(0, 1, 0)",12,-0.149823,0.801596
7,"(1, 0, -1)",7,-0.163368,0.642931
6,"(1, 0, -1)",6,-0.163368,0.642931
11,"(1, -1, 0)",11,-0.172529,0.535628
10,"(1, -1, 0)",10,-0.172529,0.535628
3,"(1, 1, 0)",3,-0.209062,0.107694
2,"(1, 1, 0)",2,-0.209062,0.107694


In [8]:
atoms_to_view = []
for i, row in df.iterrows():
    slab = study.terminations[row['Termination index']]
    atoms_to_view.append(AseAtomsAdaptor.get_atoms(slab))

view(atoms_to_view)

<Popen: returncode: None args: ['c:\\Users\\matnis\\OneDrive - Danmarks Tekn...>

In [12]:
2 in study.oer_surface_studies

False

In [ ]:


slabs = study._build_surfaces((1,0,1), stoichiometry_tolerance = 0.2,
    all_species_present = True,
    select_closest_O_stoichiometry = True,
    max_atoms=1000, max_slab_thickness=1000,
    ignore_stoichiometry_of_species = [])

atoms_list = []
for slab in slabs:
    print(study.compute_slab_thickness(slab))
    atoms_list.append(AseAtomsAdaptor.get_atoms(slab))

        
view(atoms_list)

Selected 2 slabs out of  8 based on closest O-stoichiometry.
12.874051298885838
12.772720597681072


<Popen: returncode: None args: ['c:\\Users\\matnis\\OneDrive - Danmarks Tekn...>

In [ ]:
for slab in slabs:
    print(len(slab))

48


In [ ]:
for slab in slabs:
    d = get_stoichiometry_diff(
        get_stoichiometry_dict(study.bulk_structure),
        get_stoichiometry_dict(slab)) 
    print(d['O'])
    print( eval_stoichiometry_diff(study.bulk_structure, slab,
                                max_diff_dict={'O': 0.02},   
                                )
           )
    print('_____________________________________')


0.0
True
_____________________________________
